<a href="https://colab.research.google.com/github/SmritiD2004/RAG--IBMSkillsbuild/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai sentence-transformers faiss-cpu

In [ ]:
import faiss
import numpy as np
import google.generativeai as genai

from sentence_transformers import SentenceTransformer
from google.colab import files
from google.colab import userdata

In [ ]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini initialized successfully!")

In [ ]:
user_query = input("Ask a question: ")

response = model.generate_content(user_query)

initial_response = response.text

print("\nBaseline Response (Without RAG):\n")
print(initial_response)

In [ ]:
uploaded = files.upload()

doc_path = list(uploaded.keys())[0]

with open(doc_path, "r", encoding="utf-8") as f:
    document_text = f.read()

print("Document Loaded Successfully!")

In [ ]:
def split_into_chunks(text, chunk_size=300, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_into_chunks(document_text)

print(f"Document split into {len(chunks)} chunks.")

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(chunks)

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

stored_chunks = chunks

print(f"Stored {len(stored_chunks)} chunks in FAISS.")

In [ ]:
query_embedding = embed_model.encode([user_query])

k = 3

distances, indices = index.search(np.array(query_embedding), k)

retrieved_context = "\n\n".join(
    [stored_chunks[i] for i in indices[0]]
)

print("Retrieved Context:\n")
print(retrieved_context)

In [ ]:
prompt = f"""
You are a helpful travel assistant.

Answer ONLY using the information provided in the context below.

If the answer is not present in the context, say:
"I don't have enough information in the provided documents."

Context:
{retrieved_context}

Question:
{user_query}
"""

response = model.generate_content(prompt)

rag_response = response.text

print("\nRAG Response:\n")
print(rag_response)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"""
# Original Query

**{user_query}**

---

# Without RAG

{initial_response}

---

# With RAG

{rag_response}
"""))